In [ ]:
# Load packages
import re

import pandas as pd

In [ ]:
# Load the stata data into a data-frame
file_path = "data/ESA1821_neu_SO.dta"  # stata file location

df = pd.read_stata(file_path)
print(f"Data-frame shape: {df.shape}")
print("Variable names: " + ", ".join(df.columns))

In [ ]:
# Get descriptions and information for variables from the stat log-file
logfile_path = "data/codebook_output.txt"  # stata logfile location

with open(logfile_path, "r", encoding="utf-8") as f:
    stata_logfile = f.read()

vars_info: dict[str, tuple[str, str]] = {}  # Mapping from variable names to tuple of description and information
for var_name in df.columns:
    info_header = r"\{txt\}\{hline\}\n\{res\}" + var_name + r"\{right:([^\}]+)\}\n\{txt\}\{hline\}"
    info_footer = r"\n\n(?:\{txt\}\{hline\}|\{com\})"
    match_var = re.search(rf"{info_header}(.*?){info_footer}", stata_logfile, flags=re.DOTALL)
    if match_var is not None and len(groups := match_var.groups()) == 2:
        group1, group2 = groups
        var_description, var_body = group1 if isinstance(group1, str) else "", group2 if isinstance(group2, str) else ""
    else:
        var_description, var_body = "", ""
    var_long_info = f"== {var_name} =={{col 26}}" + var_description + var_body

    indented_lines: list[str] = []
    for line in var_long_info.split("\n"):
        indented_line = ""
        parts = re.split(r"(\{col \d+\})", line)
        for part in parts:
            match_indentation = re.match(r"\{col (\d+)\}", part)
            if match_indentation:
                indented_line += " " * (int(match_indentation.group(1)) - 1 - len(indented_line))
            else:
                indented_line += re.sub(r"\{([^\}]+?)(:([^\}]+))?\}", r"\3", part)
        indented_lines.append(indented_line)
    var_long_info = "\n".join(indented_lines)

    vars_info[var_name] = var_description, var_long_info  # tuple of description and information

print("Variables: " + " ● ".join(vars_info[var][0] for var in df.columns))

In [ ]:
# Get info about any variable
print("Description:\n\t" + vars_info[df.columns[19]][0], end="\n\n\n\n")
print("Information:\n\t" + vars_info[df.columns[19]][1])

In [ ]:
# Inspect the time variable
print(vars_info["zeit"][1])  # data is from 2018 and 2021
# df_2018 = df.query("zeit == 2018")
# df_2021 = df.query("zeit == 2021")

In [ ]:
# Partition variables into groups
idk_ids = [3, 4, 46, *range(107, 109 + 1), 116, 155]
general_ids = [*range(0, 2 + 1), 5]
socioecon_ids = [*range(6, 18 + 1), *range(173, 174 + 1)]
alcohol_ids = [*range(19, 45 + 1), *range(47, 59 + 1), 175]
nicotine_ids = [*range(60, 72 + 1), *range(176, 178 + 1)]
other_drugs_ids = [*range(73, 106 + 1), *range(110, 115 + 1), *range(117, 121 + 1)]
medication_ids = [*range(122, 154 + 1), *range(156, 172 + 1)]

idk_vars = [df.columns[i] for i in idk_ids]
general_vars = [df.columns[i] for i in general_ids]
socioecon_vars = [df.columns[i] for i in socioecon_ids]
alcohol_vars = [df.columns[i] for i in alcohol_ids]
nicotine_vars = [df.columns[i] for i in nicotine_ids]
other_drugs_vars = [df.columns[i] for i in other_drugs_ids]
medication_vars = [df.columns[i] for i in medication_ids]

for group_name, var_names in zip(
    ["Miscellaneous", "General", "Socioeconomic", "Alcohol", "Nicotine", "Other drug", "Medication"],
    [idk_vars, general_vars, socioecon_vars, alcohol_vars, nicotine_vars, other_drugs_vars, medication_vars],
):
    print(f"{group_name} variables:  " + " ● ".join(vars_info[var][0] for var in var_names), end="\n\n")